# Dynamics of the Interplanetary Gas and Magnetic Fields (Parker, 1958): Implementation / 구현

Parker의 태양풍 이론의 핵심 수학을 직접 구현합니다.

This notebook implements the core mathematics of Parker's solar wind theory.

**구현 내용 / Contents:**
1. 정수압 평형의 실패 — 무한대 압력 / Hydrostatic Failure — Pressure at Infinity
2. Parker 등온 태양풍 해 — Figure 1 재현 / Parker Isothermal Solution — Figure 1 Reproduction
3. 임계점 분석 — de Laval 노즐 유사성 / Critical Point Analysis — de Laval Nozzle Analogy
4. 코로나 온도에 따른 태양풍 매개변수 / Solar Wind Parameters vs. Coronal Temperature
5. Parker Spiral 자기장 시각화 / Parker Spiral Magnetic Field Visualization
6. 태양풍 밀도·속도·자기장의 방사 프로파일 / Radial Profiles of Wind Density, Velocity, and Field
7. 종합 — 태양풍이 지구에 미치는 영향 / Synthesis — Solar Wind Impact at Earth

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "font.size": 11,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

# Physical constants
G = 6.674e-8        # Gravitational constant (CGS: cm³/g/s²)
M_sun = 1.989e33    # Solar mass (g)
M_H = 1.673e-24     # Hydrogen mass (g)
k_B = 1.381e-16     # Boltzmann constant (erg/K)
R_sun = 6.96e10     # Solar radius (cm)
AU = 1.496e13       # 1 AU (cm)
a = 1e11            # Coronal base radius (cm) — Parker's choice ~10^6 km

## Part 1: 정수압 평형의 실패 / Hydrostatic Failure

Chapman (1957)의 정적 코로나 모델에서는 $p(\infty) = p_0 \exp[-\lambda(n+1)/n] \neq 0$.
이 유한한 잔여 압력을 상쇄할 성간 압력이 없으므로 정수압 평형은 불가능합니다.

In Chapman's (1957) static model, $p(\infty) \neq 0$ — with no interstellar pressure to balance it, hydrostatic equilibrium is impossible.

In [ ]:
def compute_lambda(T0: float, a_base: float = a) -> float:
    """Compute dimensionless gravity parameter lambda.

    Args:
        T0: Coronal base temperature (K).
        a_base: Coronal base radius (cm).

    Returns:
        Dimensionless lambda = GM_sun * M_H / (2 * a * k_B * T0).
    """
    return G * M_sun * M_H / (2 * a_base * k_B * T0)


def hydrostatic_pressure_ratio(T0: float, n: float = 2.5) -> float:
    """Compute p(infinity)/p0 for hydrostatic equilibrium (eq. 9).

    Args:
        T0: Coronal temperature (K).
        n: Heat conduction exponent (2.5 for ionized H, 0.5 for neutral H).

    Returns:
        p(inf)/p0 ratio.
    """
    lam = compute_lambda(T0)
    return np.exp(-lam * (n + 1) / n)


# Compute for range of temperatures
T0_range = np.linspace(0.5e6, 4e6, 200)
p_ratio_ionized = [hydrostatic_pressure_ratio(T, n=2.5) for T in T0_range]
p_ratio_neutral = [hydrostatic_pressure_ratio(T, n=0.5) for T in T0_range]
lambda_vals = [compute_lambda(T) for T in T0_range]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# (a) Lambda vs T0
ax = axes[0]
ax.plot(T0_range / 1e6, lambda_vals, "k-", linewidth=2)
ax.axhline(2, color="r", linestyle="--", label=r"$\lambda = 2$ (supersonic threshold)")
ax.set_xlabel(r"Coronal Temperature $T_0$ ($\times 10^6$ K)")
ax.set_ylabel(r"$\lambda = GM_\odot M / 2akT_0$")
ax.set_title(r"(a) Dimensionless Gravity Parameter $\lambda$ / 무차원 중력 매개변수")
ax.legend()

# (b) p(inf)/p0
ax = axes[1]
ax.semilogy(T0_range / 1e6, p_ratio_ionized, "b-", linewidth=2, label="Ionized H (n=5/2)")
ax.semilogy(T0_range / 1e6, p_ratio_neutral, "r--", linewidth=2, label="Neutral H (n=1/2)")
ax.axhline(1e-13 / (2 * 3e7 * k_B * 1.5e6), color="g", linestyle=":",
           label=r"$p_{\rm ISM}/p_0$ (interstellar)")
ax.set_xlabel(r"Coronal Temperature $T_0$ ($\times 10^6$ K)")
ax.set_ylabel(r"$p(\infty) / p_0$")
ax.set_title("(b) Pressure at Infinity / 무한대에서의 압력")
ax.legend(fontsize=9)
ax.set_ylim(1e-12, 1)

fig.suptitle("Part 1: Hydrostatic Equilibrium Failure / 정수압 평형의 실패", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# Key numbers from Parker's paper
T0_std = 1.5e6
lam_std = compute_lambda(T0_std)
p_inf = hydrostatic_pressure_ratio(T0_std, n=2.5)
print(f"T0 = {T0_std/1e6:.1f} × 10⁶ K:")
print(f"  λ = {lam_std:.2f}")
print(f"  p(∞)/p₀ = {p_inf:.2e}  (Parker: 0.55 × 10⁻³)")
print(f"  → p(∞) is FINITE but no ISM pressure to balance it → expansion inevitable!")

## Part 2: Parker 등온 태양풍 해 — Figure 1 재현 / Parker Isothermal Solution — Figure 1

Parker의 eq. 14를 수치적으로 풀어 Figure 1을 재현합니다. 등온 코로나($\tau = 1$)에서:

$$\psi - \ln\psi = \psi_0 - \ln\psi_0 + 4\ln\xi - 2\lambda\left(1 - \frac{1}{\xi}\right)$$

임계점 조건 ($\psi = 1$ at $\xi = \lambda/2$)으로부터 $\psi_0$를 결정하고, 각 $\xi$에서 $\psi$를 구합니다.

In [ ]:
def parker_solution(
    T0: float,
    xi_max: float = 300.0,
    n_points: int = 2000,
    a_base: float = a,
) -> tuple[np.ndarray, np.ndarray]:
    """Solve Parker's isothermal solar wind equation (eq. 14).

    Finds the unique transonic solution passing through the critical point.

    Args:
        T0: Coronal temperature (K).
        xi_max: Maximum r/a to compute.
        n_points: Number of radial points.
        a_base: Coronal base radius (cm).

    Returns:
        Tuple of (xi_array, v_array_km_s) — radius in units of a, velocity in km/s.
    """
    lam = compute_lambda(T0, a_base)
    xi_c = lam / 2  # Critical point

    if xi_c < 1.0:
        xi_c = 1.001  # Ensure critical point is outside base

    # At critical point: psi = 1 (v = sound speed)
    # Y function: Y(xi) = 4*ln(xi) - 2*lambda*(1 - 1/xi)
    # At critical point: Y_c = 4*ln(xi_c) - 2*lambda*(1 - 1/xi_c)
    # Z = psi - ln(psi), Z_c = 1 - 0 = 1
    # psi_0 - ln(psi_0) = 1 - Y_c + Y(1) = 1 - Y_c

    Y_c = 4 * np.log(xi_c) - 2 * lam * (1 - 1.0 / xi_c)

    # RHS of eq. 14 at xi=1: psi_0 - ln(psi_0) = 1 - Y_c
    rhs_at_base = 1 - Y_c  # = 1 - 4*ln(lam/2) + 2*lam - 2 = 2*lam - 3 - 4*ln(lam/2) + 2
    # Actually: at base xi=1, Y(1) = 0, so:
    # psi_0 - ln(psi_0) = (psi_c - ln(psi_c)) - Y_c + Y(1) = 1 - Y_c
    target = 1.0 - Y_c

    # Solve psi - ln(psi) = target for psi < 1 (subsonic branch at base)
    def f_psi(psi):
        return psi - np.log(psi) - target

    # For subsonic solution: psi_0 < 1
    try:
        psi_0 = brentq(f_psi, 1e-10, 1.0 - 1e-10)
    except ValueError:
        return np.array([]), np.array([])

    # Now solve for psi(xi) at each xi
    xi_arr = np.linspace(1.0, xi_max, n_points)
    psi_arr = np.zeros(n_points)

    for i, xi in enumerate(xi_arr):
        Y_xi = 4 * np.log(xi) - 2 * lam * (1 - 1.0 / xi)
        rhs = target + Y_xi  # psi - ln(psi) = target + Y(xi)

        # Choose correct branch: subsonic (psi < 1) for xi < xi_c,
        # supersonic (psi > 1) for xi > xi_c
        if xi < xi_c - 0.01:
            try:
                psi_arr[i] = brentq(f_solve, 1e-10, 1.0 - 1e-10, args=(rhs,))
            except ValueError:
                psi_arr[i] = np.nan
        elif xi > xi_c + 0.01:
            try:
                psi_arr[i] = brentq(f_solve, 1.0 + 1e-10, 500.0, args=(rhs,))
            except ValueError:
                psi_arr[i] = np.nan
        else:
            psi_arr[i] = 1.0  # At critical point

    # Convert psi to velocity in km/s
    # psi = M_H * v^2 / (2 * k_B * T0) → v = sqrt(2 * k_B * T0 * psi / M_H)
    v_arr = np.sqrt(2 * k_B * T0 * psi_arr / M_H)  # cm/s
    v_arr_km = v_arr / 1e5  # km/s

    return xi_arr, v_arr_km


def f_solve(psi: float, rhs: float) -> float:
    """Equation psi - ln(psi) = rhs to solve for psi."""
    return psi - np.log(psi) - rhs


# Reproduce Parker's Figure 1
temperatures = [0.5e6, 1.0e6, 1.5e6, 2.0e6, 2.5e6, 3.0e6, 4.0e6]
colors = plt.cm.plasma(np.linspace(0.1, 0.9, len(temperatures)))

fig, ax = plt.subplots(figsize=(12, 8))

for T0, color in zip(temperatures, colors):
    xi_arr, v_arr = parker_solution(T0, xi_max=100)
    if len(xi_arr) > 0:
        ax.plot(xi_arr, v_arr, color=color, linewidth=2,
                label=f"$T_0 = {T0/1e6:.1f} \\times 10^6$ K")

# Mark Earth's orbit
xi_earth = AU / a
ax.axvline(xi_earth, color="blue", linestyle=":", alpha=0.5)
ax.text(xi_earth + 1, 850, "Earth\n(1 AU)", fontsize=10, color="blue")

ax.set_xlabel(r"$\xi = r/a$", fontsize=13)
ax.set_ylabel("Expansion Velocity $v(r)$ (km/s)", fontsize=13)
ax.set_title("Parker's Figure 1: Isothermal Solar Wind Solutions\n"
             "Parker의 Figure 1 재현: 등온 태양풍 해", fontsize=14, fontweight="bold")
ax.legend(fontsize=10)
ax.set_xlim(1, 100)
ax.set_ylim(0, 950)

plt.tight_layout()
plt.show()

# Print velocities at Earth's orbit
print("Velocity at Earth's orbit (1 AU):")
for T0 in temperatures:
    xi_arr, v_arr = parker_solution(T0)
    if len(xi_arr) > 0:
        idx = np.argmin(np.abs(xi_arr - xi_earth))
        print(f"  T0 = {T0/1e6:.1f} × 10⁶ K: v = {v_arr[idx]:.0f} km/s")

## Part 3: 임계점 분석 — de Laval 노즐 유사성 / Critical Point Analysis — de Laval Nozzle Analogy

Parker 방정식의 수학적 구조는 de Laval 노즐과 동일합니다.
$Y(\xi) = 4\ln\xi - 2\lambda(1 - 1/\xi)$와 $Z(\psi) = \psi - \ln\psi$의 관계를 시각화합니다.
임계점($Y$ 최솟값 = $Z$ 최솟값)에서만 아음속 → 초음속 전이가 가능합니다.

The mathematical structure is identical to a de Laval nozzle. Only at the critical point ($Y$ minimum = $Z$ minimum) can the subsonic → supersonic transition occur.

In [ ]:
T0_example = 1.5e6
lam = compute_lambda(T0_example)
xi_c = lam / 2

# Y and Z functions
xi_plot = np.linspace(1, 30, 500)
Y_func = 4 * np.log(xi_plot) - 2 * lam * (1 - 1.0 / xi_plot)
Y_min = 4 * np.log(xi_c) - 2 * lam * (1 - 1.0 / xi_c)

psi_plot = np.linspace(0.01, 8, 500)
Z_func = psi_plot - np.log(psi_plot)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# (a) Y(xi)
ax = axes[0]
ax.plot(xi_plot, Y_func, "b-", linewidth=2)
ax.axvline(xi_c, color="r", linestyle="--", label=f"$\\xi_c = \\lambda/2 = {xi_c:.1f}$")
ax.axhline(Y_min, color="gray", linestyle=":", alpha=0.5)
ax.plot(xi_c, Y_min, "ro", markersize=10, zorder=5)
ax.set_xlabel(r"$\xi = r/a$")
ax.set_ylabel(r"$Y(\xi) = 4\ln\xi - 2\lambda(1-1/\xi)$")
ax.set_title(r"(a) $Y(\xi)$ — minimum at critical point / 임계점에서 최솟값")
ax.legend()

# (b) Z(psi)
ax = axes[1]
ax.plot(psi_plot, Z_func, "g-", linewidth=2)
ax.axvline(1.0, color="r", linestyle="--", label=r"$\psi = 1$ ($v = c_s$)")
ax.plot(1.0, 1.0, "ro", markersize=10, zorder=5)
ax.set_xlabel(r"$\psi = Mv^2/2kT_0$")
ax.set_ylabel(r"$Z(\psi) = \psi - \ln\psi$")
ax.set_title(r"(b) $Z(\psi)$ — minimum at sound speed / 음속에서 최솟값")
ax.legend()

# (c) All 4 solution branches in the (xi, psi) plane
ax = axes[2]
xi_arr_fine = np.linspace(1.0, 25, 1000)
# For each xi, find all roots of psi - ln(psi) = target + Y(xi)
# Target = 1 - Y_c (transonic solution)
target = 1.0 - Y_min

for xi in xi_arr_fine:
    Y_xi = 4 * np.log(xi) - 2 * lam * (1 - 1.0 / xi)
    rhs = target + Y_xi

    # Subsonic branch (psi < 1)
    try:
        psi_sub = brentq(f_solve, 1e-10, 1.0 - 1e-10, args=(rhs,))
        ax.plot(xi, psi_sub, "b.", markersize=0.5)
    except ValueError:
        pass

    # Supersonic branch (psi > 1)
    try:
        psi_sup = brentq(f_solve, 1.0 + 1e-10, 100.0, args=(rhs,))
        ax.plot(xi, psi_sup, "r.", markersize=0.5)
    except ValueError:
        pass

# Also plot non-physical solutions (breeze and accretion)
for offset in [-1.0, -0.5, 0.5, 1.0]:
    target_off = target + offset
    psi_low_arr = []
    psi_high_arr = []
    xi_arr_off = []
    for xi in xi_arr_fine:
        Y_xi = 4 * np.log(xi) - 2 * lam * (1 - 1.0 / xi)
        rhs = target_off + Y_xi
        try:
            p_low = brentq(f_solve, 1e-10, 1.0 - 1e-10, args=(rhs,))
            psi_low_arr.append(p_low)
            xi_arr_off.append(xi)
        except ValueError:
            pass
    if psi_low_arr:
        ax.plot(xi_arr_off, psi_low_arr, "gray", linewidth=0.5, alpha=0.4)

ax.axhline(1.0, color="k", linestyle=":", alpha=0.3)
ax.plot(xi_c, 1.0, "k*", markersize=15, zorder=5, label="Critical point")
ax.set_xlabel(r"$\xi = r/a$")
ax.set_ylabel(r"$\psi = Mv^2/2kT_0$")
ax.set_title("(c) Solution Topology / 해의 위상 구조")
ax.set_ylim(0, 6)
ax.legend()

# Labels
ax.annotate("Transonic\n(solar wind)", xy=(15, 3.5), fontsize=10, color="r", fontweight="bold")
ax.annotate("Subsonic\n(breeze)", xy=(15, 0.3), fontsize=10, color="b", fontweight="bold")

fig.suptitle(f"Part 3: Critical Point Analysis ($T_0 = {T0_example/1e6:.1f} \\times 10^6$ K, "
             f"$\\lambda = {lam:.2f}$) / 임계점 분석",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"Critical point: r_c = {xi_c:.1f}a = {xi_c * a / R_sun:.1f} R_sun")
print(f"Sound speed at T0 = {T0_example/1e6:.1f} MK: c_s = {np.sqrt(2*k_B*T0_example/M_H)/1e5:.0f} km/s")

## Part 4: 코로나 온도에 따른 태양풍 매개변수 / Solar Wind Parameters vs. Coronal Temperature

코로나 온도 $T_0$가 태양풍의 모든 것을 결정합니다: 속도, 밀도, 질량 손실률, 임계점 위치.

$T_0$ determines everything: velocity, density, mass loss rate, critical point location.

In [ ]:
T0_scan = np.linspace(0.5e6, 4e6, 50)
v_earth = []
r_critical = []
N0 = 3e7  # Base density (cm^-3) from Parker

for T0 in T0_scan:
    xi_arr, v_arr = parker_solution(T0, xi_max=200)
    if len(xi_arr) > 0:
        idx = np.argmin(np.abs(xi_arr - xi_earth))
        v_earth.append(v_arr[idx])
    else:
        v_earth.append(np.nan)
    lam = compute_lambda(T0)
    r_critical.append(lam / 2)

v_earth = np.array(v_earth)

# Compute density and mass loss at Earth
# From continuity: N(r)*v(r) = N0*v0*(a/r)^2
# v0 from Parker solution at xi=1
v0_arr = []
for T0 in T0_scan:
    xi_arr, v_arr = parker_solution(T0, xi_max=200)
    if len(xi_arr) > 0:
        v0_arr.append(v_arr[0])
    else:
        v0_arr.append(np.nan)
v0_arr = np.array(v0_arr)

# N at Earth = N0 * v0 * (a/r_earth)^2 / v_earth
N_earth = N0 * (v0_arr * 1e5) * (a / AU)**2 / (v_earth * 1e5)  # cm^-3
# Mass loss: dM/dt = 4*pi*a^2 * N0 * M_H * v0
mass_loss = 4 * np.pi * a**2 * N0 * M_H * (v0_arr * 1e5)  # g/s

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# (a) Velocity at Earth
ax = axes[0, 0]
ax.plot(T0_scan / 1e6, v_earth, "r-", linewidth=2)
ax.axhspan(400, 800, alpha=0.1, color="green", label="Observed range (400–800 km/s)")
ax.set_xlabel(r"$T_0$ ($\times 10^6$ K)")
ax.set_ylabel("v at 1 AU (km/s)")
ax.set_title("(a) Velocity at Earth / 지구 궤도 속도")
ax.legend()

# (b) Critical point
ax = axes[0, 1]
ax.plot(T0_scan / 1e6, r_critical, "b-", linewidth=2)
r_crit_rsun = np.array(r_critical) * a / R_sun
ax2 = ax.twinx()
ax2.plot(T0_scan / 1e6, r_crit_rsun, "b-", linewidth=0, alpha=0)
ax2.set_ylabel(r"$r_c$ ($R_\odot$)", color="gray")
ax.set_xlabel(r"$T_0$ ($\times 10^6$ K)")
ax.set_ylabel(r"$r_c / a$")
ax.set_title("(b) Critical Point Radius / 임계점 반경")

# (c) Density at Earth
ax = axes[1, 0]
ax.semilogy(T0_scan / 1e6, N_earth, "g-", linewidth=2)
ax.axhspan(3, 20, alpha=0.1, color="orange", label="Observed range (3–20 cm⁻³)")
ax.set_xlabel(r"$T_0$ ($\times 10^6$ K)")
ax.set_ylabel("N at 1 AU (cm⁻³)")
ax.set_title("(c) Density at Earth / 지구 궤도 밀도")
ax.legend()

# (d) Mass loss rate
ax = axes[1, 1]
ax.semilogy(T0_scan / 1e6, mass_loss, "m-", linewidth=2)
ax.axhline(1e14, color="k", linestyle="--", alpha=0.5, label=r"Biermann: $10^{14}$ g/s")
ax.set_xlabel(r"$T_0$ ($\times 10^6$ K)")
ax.set_ylabel(r"$dM/dt$ (g/s)")
ax.set_title("(d) Mass Loss Rate / 질량 손실률")
ax.legend()

fig.suptitle("Part 4: Solar Wind Parameters vs. Coronal Temperature / 코로나 온도별 태양풍 매개변수",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## Part 5: Parker Spiral 자기장 시각화 / Parker Spiral Magnetic Field Visualization

태양 자전($\omega \simeq 2.7 \times 10^{-6}$ rad/s)과 방사상 태양풍($v_m$)에 의해 자기장선이 아르키메데스 나선을 형성합니다.

$$B_r = B_0\left(\frac{b}{r}\right)^2, \quad B_\phi = B_0\frac{\omega(r-b)}{v_m}\left(\frac{b}{r}\right)^2\sin\theta$$

Solar rotation + radial wind → field lines form Archimedean spirals (Parker spiral).

In [ ]:
def parker_spiral_fieldline(
    phi0: float,
    v_m: float = 400e5,
    omega: float = 2.7e-6,
    b: float = 5e11,
    r_max: float = 8 * AU,
    n_points: int = 2000,
) -> tuple[np.ndarray, np.ndarray]:
    """Compute a Parker spiral field line in the equatorial plane.

    Args:
        phi0: Initial azimuthal angle (rad).
        v_m: Radial wind speed (cm/s).
        omega: Solar angular velocity (rad/s).
        b: Source surface radius (cm).
        r_max: Maximum radius (cm).
        n_points: Number of points.

    Returns:
        Tuple of (r_array, phi_array) in cm and radians.
    """
    r = np.linspace(b, r_max, n_points)
    # From eq. 25: r/b - 1 - ln(r/b) = v_m/(b*omega) * (phi - phi0)
    phi = phi0 + (b * omega / v_m) * (r / b - 1 - np.log(r / b))
    return r, phi


fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# (a) Parker spiral for different wind speeds
ax = axes[0]
v_winds = [300e5, 400e5, 600e5, 800e5]  # cm/s
colors_sp = ["blue", "green", "orange", "red"]
labels_sp = ["300 km/s (slow)", "400 km/s", "600 km/s", "800 km/s (fast)"]

for v_m, color, label in zip(v_winds, colors_sp, labels_sp):
    for phi0 in np.linspace(0, 2 * np.pi, 4, endpoint=False):
        r, phi = parker_spiral_fieldline(phi0, v_m=v_m)
        x = r * np.cos(phi) / AU
        y = r * np.sin(phi) / AU
        lbl = label if phi0 == 0 else None
        ax.plot(x, y, color=color, linewidth=1, alpha=0.7, label=lbl)

# Mark planets
for name, dist in [("Mercury", 0.39), ("Venus", 0.72), ("Earth", 1.0), ("Mars", 1.52)]:
    circle = plt.Circle((0, 0), dist, fill=False, color="gray", linestyle=":", linewidth=0.5)
    ax.add_patch(circle)
    ax.text(dist * 0.7, dist * 0.7, name, fontsize=8, color="gray")

ax.plot(0, 0, "yo", markersize=15, zorder=5)  # Sun
ax.set_xlim(-5.5, 5.5)
ax.set_ylim(-5.5, 5.5)
ax.set_aspect("equal")
ax.set_xlabel("x (AU)")
ax.set_ylabel("y (AU)")
ax.set_title("(a) Parker Spiral — Different Wind Speeds\n(a) 파커 나선 — 태양풍 속도별")
ax.legend(fontsize=9, loc="upper right")

# (b) Spiral angle vs distance
ax = axes[1]
r_au = np.linspace(0.1, 5, 500)
r_cm = r_au * AU
omega = 2.7e-6
b = 5e11

for v_m, color, label in zip(v_winds, colors_sp, labels_sp):
    # tan(spiral_angle) = B_phi / B_r = omega*(r-b)/v_m
    angle = np.degrees(np.arctan(omega * (r_cm - b) / v_m))
    ax.plot(r_au, angle, color=color, linewidth=2, label=label)

ax.axhline(45, color="k", linestyle=":", alpha=0.3, label="45°")
ax.axvline(1.0, color="gray", linestyle="--", alpha=0.3)
ax.set_xlabel("Distance (AU)")
ax.set_ylabel("Spiral Angle (degrees)")
ax.set_title("(b) Spiral Angle vs Distance / 나선각 vs 거리")
ax.legend(fontsize=9)
ax.set_ylim(0, 90)

fig.suptitle("Part 5: Parker Spiral / 파커 나선 자기장", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# Print spiral angle at 1 AU
for v_m, label in zip(v_winds, labels_sp):
    angle_earth = np.degrees(np.arctan(omega * (AU - b) / v_m))
    print(f"  {label}: spiral angle at 1 AU = {angle_earth:.0f}°")

## Part 6: 태양풍 방사 프로파일 / Radial Profiles of Solar Wind

속도, 밀도, 자기장이 거리에 따라 어떻게 변하는지 종합적으로 시각화합니다.
연속 방정식에 의해 $N \propto v^{-1} r^{-2}$, Parker spiral에 의해 $B_r \propto r^{-2}$, $B_\phi \propto r^{-1}$.

Comprehensive visualization of how velocity, density, and magnetic field vary with distance.

In [ ]:
T0_ref = 1.5e6
xi_arr, v_arr = parker_solution(T0_ref, xi_max=300)
r_arr_au = xi_arr * a / AU

# Density from continuity: N*v*r^2 = const
v0_ref = v_arr[0] * 1e5  # cm/s
N_arr = N0 * v0_ref * (a / (xi_arr * a))**2 / (v_arr * 1e5)  # cm^-3

# Magnetic field (Parker spiral, equatorial plane theta = pi/2)
B0_surface = 1.0  # Gauss at source surface
b_source = 5e11  # Source surface radius (cm)
r_arr_cm = xi_arr * a
B_r_arr = B0_surface * (b_source / r_arr_cm)**2  # Gauss
omega_sun = 2.7e-6
v_m_ref = 400e5  # cm/s
B_phi_arr = B0_surface * (omega_sun * (r_arr_cm - b_source) / v_m_ref) * (b_source / r_arr_cm)**2
B_total = np.sqrt(B_r_arr**2 + B_phi_arr**2)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# (a) Velocity
ax = axes[0, 0]
ax.plot(r_arr_au, v_arr, "r-", linewidth=2)
ax.axhline(np.sqrt(2 * k_B * T0_ref / M_H) / 1e5, color="gray", linestyle="--",
           label=f"Sound speed = {np.sqrt(2*k_B*T0_ref/M_H)/1e5:.0f} km/s")
ax.axvline(1.0, color="blue", linestyle=":", alpha=0.3)
ax.set_xlabel("Distance (AU)")
ax.set_ylabel("Velocity (km/s)")
ax.set_title("(a) Solar Wind Velocity / 태양풍 속도")
ax.set_xlim(0, 2)
ax.legend()

# (b) Density
ax = axes[0, 1]
ax.semilogy(r_arr_au, N_arr, "g-", linewidth=2)
ax.axvline(1.0, color="blue", linestyle=":", alpha=0.3)
ax.set_xlabel("Distance (AU)")
ax.set_ylabel("Density (cm⁻³)")
ax.set_title("(b) Solar Wind Density / 태양풍 밀도")
ax.set_xlim(0, 2)

# (c) Magnetic field components
ax = axes[1, 0]
mask = r_arr_au > 0.05
ax.semilogy(r_arr_au[mask], np.abs(B_r_arr[mask]) * 1e5, "b-", linewidth=2, label=r"$|B_r|$")
ax.semilogy(r_arr_au[mask], np.abs(B_phi_arr[mask]) * 1e5, "r--", linewidth=2, label=r"$|B_\phi|$")
ax.semilogy(r_arr_au[mask], B_total[mask] * 1e5, "k-", linewidth=1, alpha=0.5, label=r"$|B|$")
ax.axvline(1.0, color="blue", linestyle=":", alpha=0.3)
ax.set_xlabel("Distance (AU)")
ax.set_ylabel("Magnetic Field (nT)")
ax.set_title(r"(c) IMF Components: $B_r \propto r^{-2}$, $B_\phi \propto r^{-1}$ / 자기장 성분")
ax.set_xlim(0.05, 5)
ax.legend()

# (d) Dynamic pressure vs magnetic pressure
ax = axes[1, 1]
p_dyn = 0.5 * N_arr * M_H * (v_arr * 1e5)**2  # dynes/cm^2
p_mag = B_total**2 / (8 * np.pi)  # dynes/cm^2
ax.semilogy(r_arr_au, p_dyn, "r-", linewidth=2, label=r"$\frac{1}{2}\rho v^2$ (dynamic)")
ax.semilogy(r_arr_au, p_mag, "b--", linewidth=2, label=r"$B^2/8\pi$ (magnetic)")
ax.axvline(1.0, color="blue", linestyle=":", alpha=0.3)
ax.set_xlabel("Distance (AU)")
ax.set_ylabel("Pressure (dynes/cm²)")
ax.set_title("(d) Dynamic vs Magnetic Pressure / 동압 vs 자기압")
ax.set_xlim(0.05, 5)
ax.legend()

fig.suptitle(f"Part 6: Solar Wind Radial Profiles ($T_0 = {T0_ref/1e6:.1f} \\times 10^6$ K) / "
             "태양풍 방사 프로파일", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# Values at 1 AU
idx_earth = np.argmin(np.abs(r_arr_au - 1.0))
print(f"At 1 AU (T0 = {T0_ref/1e6:.1f} MK):")
print(f"  v = {v_arr[idx_earth]:.0f} km/s")
print(f"  N = {N_arr[idx_earth]:.1f} cm⁻³")
print(f"  |B| = {B_total[idx_earth]*1e5:.1f} nT")
print(f"  B_r = {B_r_arr[idx_earth]*1e5:.1f} nT, B_phi = {B_phi_arr[idx_earth]*1e5:.1f} nT")

## Part 7: 종합 — 태양풍이 지구에 미치는 영향 / Synthesis — Solar Wind Impact at Earth

Parker의 태양풍 이론을 이전 논문들(Chapman-Ferraro 자기권, Chapman-Bartels 지자기 지수)과 연결합니다:
태양풍 동압이 자기권계면 위치를, IMF $B_z$ 성분이 자기 폭풍 강도를 결정합니다.

Connecting Parker's wind theory to previous papers: solar wind ram pressure determines magnetopause standoff, IMF $B_z$ determines storm intensity.

In [ ]:
# From Paper #2 (Chapman-Ferraro): magnetopause standoff distance
# r_mp = r_E * (B0^2 / (2*mu0*rho*v^2))^(1/6)

B0_earth = 3.1e-5  # Earth's equatorial surface field (T)
R_earth = 6.371e8  # Earth's radius (cm)
mu0 = 4 * np.pi * 1e-1  # permeability in CGS (Gauss units)

def magnetopause_distance(v_km: float, n_cm3: float) -> float:
    """Compute magnetopause standoff distance from Chapman-Ferraro pressure balance.

    Args:
        v_km: Solar wind velocity (km/s).
        n_cm3: Solar wind density (cm^-3).

    Returns:
        Standoff distance in Earth radii.
    """
    v_cm = v_km * 1e5  # cm/s
    rho = n_cm3 * M_H  # g/cm^3
    p_dyn = 0.5 * rho * v_cm**2  # dynes/cm^2
    B0_cgs = B0_earth * 1e4  # Convert T to Gauss
    # Pressure balance: B^2/(8*pi) = p_dyn at r_mp
    # B(r) = B0 * (R_E/r)^3 → B0^2*(R_E/r_mp)^6 / (8*pi) = p_dyn
    r_mp = R_earth * (B0_cgs**2 / (8 * np.pi * p_dyn))**(1.0 / 6.0)
    return r_mp / R_earth


# Scan over coronal temperatures
T0_scan2 = np.linspace(0.5e6, 3.5e6, 30)
r_mp_arr = []

for T0 in T0_scan2:
    xi_arr_t, v_arr_t = parker_solution(T0, xi_max=200)
    if len(xi_arr_t) > 0:
        idx = np.argmin(np.abs(xi_arr_t - xi_earth))
        v_e = v_arr_t[idx]
        v0_t = v_arr_t[0] * 1e5
        n_e = N0 * v0_t * (a / AU)**2 / (v_e * 1e5)
        r_mp_arr.append(magnetopause_distance(v_e, n_e))
    else:
        r_mp_arr.append(np.nan)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# (a) Magnetopause distance vs T0
ax = axes[0]
ax.plot(T0_scan2 / 1e6, r_mp_arr, "b-", linewidth=2)
ax.axhline(10, color="g", linestyle="--", alpha=0.5, label="Typical: 10 $R_E$")
ax.set_xlabel(r"Coronal Temperature $T_0$ ($\times 10^6$ K)")
ax.set_ylabel(r"Magnetopause Distance ($R_E$)")
ax.set_title("(a) Magnetopause Standoff / 자기권계면 위치")
ax.legend()

# (b) IMF Bz from Parker spiral — connection to storms
ax = axes[1]
# At 1 AU, B_r and B_phi are comparable. B_z depends on the tilt of the
# heliospheric current sheet and the spiral geometry.
# In the Parker spiral, B is in the r-phi plane (B_theta = 0).
# The z-component (GSM) depends on the angle between the spiral plane
# and the ecliptic. We show a simplified model.
spiral_angles_deg = np.linspace(0, 90, 100)
# If the field has total magnitude B_total_1AU and spiral angle alpha,
# and is tilted by angle delta from the ecliptic:
B_total_1AU = 5.0  # nT typical
for delta in [0, 15, 30, 45]:
    Bz = -B_total_1AU * np.sin(np.radians(spiral_angles_deg)) * np.sin(np.radians(delta))
    ax.plot(spiral_angles_deg, Bz, linewidth=2,
            label=f"Tilt $\\delta = {delta}°$")
ax.axhline(0, color="k", linewidth=0.5)
ax.set_xlabel("Spiral Angle (degrees)")
ax.set_ylabel(r"$B_z$ (nT, GSM)")
ax.set_title(r"(b) IMF $B_z$ from Parker Spiral / 파커 나선의 $B_z$ 성분")
ax.legend(fontsize=9)

# (c) Summary: how Parker connects all previous papers
ax = axes[2]
ax.axis("off")
summary_text = """
╔══════════════════════════════════════════╗
║     Parker의 태양풍이 연결하는 것들        ║
║     What Parker's Solar Wind Connects    ║
╠══════════════════════════════════════════╣
║                                          ║
║  Biermann (1951)                         ║
║    혜성 꼬리 → 연속 가스 유출              ║
║         ↓                                ║
║  Parker (1958) ← 이 논문                 ║
║    정수압 실패 → 초음속 태양풍              ║
║    + Parker spiral 자기장                 ║
║         ↓                                ║
║  Chapman-Ferraro (#2)                    ║
║    연속 태양풍 → 상시 자기권               ║
║    태양풍 동압 → 자기권계면 위치            ║
║         ↓                                ║
║  Chapman-Bartels (#3)                    ║
║    27일 재현 = 코로나 홀 + 태양 자전        ║
║    Kp, Dst 지수 = 태양풍 변동의 척도       ║
║         ↓                                ║
║  Dungey (#6, 다다음 논문)                 ║
║    Parker spiral Bz 남향 → 재결합          ║
║    → 자기 폭풍의 핵심 제어 변수!           ║
║                                          ║
╚══════════════════════════════════════════╝
"""
ax.text(0.05, 0.95, summary_text, transform=ax.transAxes,
        fontsize=10, verticalalignment="top", fontfamily="monospace",
        bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.8))
ax.set_title("(c) Connections / 연결 관계")

fig.suptitle("Part 7: Solar Wind Impact at Earth / 태양풍이 지구에 미치는 영향",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## Summary / 요약

| Part | 구현 내용 / Implementation | Parker (1958)의 기여 / Contribution |
|------|--------------------------|-----------------------------------|
| 1 | 정수압 평형의 실패 — $p(\infty)/p_0$ 계산 | Chapman의 정적 코로나 반증 / Refutation of Chapman's static corona |
| 2 | Parker Figure 1 재현 — 온도별 팽창 속도 | 등온 태양풍 해의 정량적 도출 / Quantitative isothermal wind solutions |
| 3 | 임계점 분석 — $Y(\xi)$, $Z(\psi)$ 해의 위상 구조 | de Laval 노즐 유사성, 초음속 전이 / de Laval analogy, sonic transition |
| 4 | $T_0$ → 속도, 밀도, 질량 손실 관계 | 코로나 온도가 모든 것을 결정 / Coronal temperature determines everything |
| 5 | Parker spiral 시각화 — 나선각 vs 거리 | 동결 자기장 + 태양 자전 → 나선 / Frozen-in field + rotation → spiral |
| 6 | 속도, 밀도, 자기장의 방사 프로파일 | $v \uparrow$, $N \propto r^{-2}$, $B_r \propto r^{-2}$, $B_\phi \propto r^{-1}$ |
| 7 | 자기권계면 위치, IMF $B_z$, 논문 간 연결 | Parker → Chapman-Ferraro → Dungey 연결의 완성 |

**다음 논문 / Next paper**: Van Allen et al. (1958) — 방사선대 발견. Parker의 연속 태양풍이 자기권에 에너지를 공급하여 방사선대의 입자를 유지합니다.